## Code Generation

In [67]:
import os
import io
import sys
from openai import OpenAI
import anthropic
import google.generativeai as genai

In [79]:
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', '****')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', '****')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY','***')

In [69]:
openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"
GEMINI_MODEL = "gemini-2.0-flash-exp"
genai.configure()

In [70]:
system_messages = "You are an assistant that reimplements Python code in high performance C++ for an M1 Mac. "
system_messages += "Respond only with C++ code; use comments sparingly and do not provide any explanation other than occasional comments. "
system_messages += "The C++ response needs to produce an identical output in the fastest possible time."

In [71]:
def user_prompt_for(python):
    user_prompt = "Rewrite this Python code in C++ with the fastest possible implementation that produces identical output in the least time. "
    user_prompt += "Respond only with C++ code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to number types to ensure no int overflows. Remember to #include all necessary C++ packages such as iomanip.\n\n"
    user_prompt += python
    return user_prompt

In [72]:
def messages_for(python):
    return [
        {"role":"system", "content": system_messages},
        {"role": "user", "content": user_prompt_for(python) }
    ]

In [73]:

def write_output(cpp, file_name):
    code = cpp.replace("```cpp","").replace("```","")
    with open(file_name, "w") as f:
        f.write(code)

In [74]:
def optimize_gpt(python):
    messages = messages_for(python)
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages, stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply, "optimzed_new_gpt.cpp")
    

In [75]:
def optimize_gemini(python):
    model = genai.GenerativeModel(GEMINI_MODEL)
    response_stream = model.generate_content(user_prompt_for(python), stream = True)

    response = ""
    for chunk in response_stream:
        if chunk.text:
            response += chunk.text or ""
            print(chunk.text, end='', flush=True)  
    write_output(response, "optimzed_new_gemini.cpp")

In [76]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [77]:
optimize_gpt(pi)

```cpp
#include <iostream>
#include <chrono>
#include <iomanip>

double calculate(long long iterations, double param1, double param2) {
    double result = 1.0;
    for (long long i = 1; i <= iterations; ++i) {
        double j = i * param1 - param2;
        result -= (1.0 / j);
        j = i * param1 + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    // Start time measurement
    auto start = std::chrono::high_resolution_clock::now();

    double result = calculate(100000000LL, 4.0, 1.0) * 4;

    // End time measurement
    auto end = std::chrono::high_resolution_clock::now();

    // Calculate execution time in seconds
    std::chrono::duration<double> execution_time = end - start;
    
    // Output result and execution time
    std::cout << "Result: " << std::fixed << std::setprecision(12) << result << "\n";
    std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << execution_time.count() << " seconds\n";

    return 0;
}
```

In [78]:
optimize_gemini(pi)

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>

using namespace std;

double calculate(long long iterations, int param1, int param2) {
    double result = 1.0;
    for (long long i = 1; i <= iterations; ++i) {
        double j = static_cast<double>(i * param1) - param2;
        result -= (1.0 / j);
        j = static_cast<double>(i * param1) + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    auto start_time = chrono::high_resolution_clock::now();
    double result = calculate(100000000LL, 4, 1) * 4;
    auto end_time = chrono::high_resolution_clock::now();

    auto duration = chrono::duration_cast<chrono::microseconds>(end_time - start_time);
    double execution_time = duration.count() / 1000000.0;

    cout << fixed << setprecision(12) << "Result: " << result << endl;
    cout << fixed << setprecision(6) << "Execution Time: " << execution_time << " seconds" << endl;

    return 0;
}
```